In [1]:
import os
import torch
import numpy as np
from PIL import Image
from tqdm import tqdm
from transformers import AutoImageProcessor, MaskFormerForInstanceSegmentation
import glob
from datetime import datetime

# --- Configuration ---
MODEL_ID = "facebook/maskformer-resnet101-cityscapes"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Adjust DATA_ROOT to be more flexible (Matches SegFormer reference)
DATA_ROOT = os.path.join("cityscapes")
IMG_DIR = os.path.join(DATA_ROOT, "leftImg8bit", "val")
GT_DIR = os.path.join(DATA_ROOT, "gtFine", "val")

# --- Memory optimization settings ---
MAX_SAMPLES = None  
EVAL_SIZE = (1024, 512) 

ID2LABEL = {
    0: "road", 1: "sidewalk", 2: "building", 3: "wall", 4: "fence", 5: "pole", 
    6: "traffic light", 7: "traffic sign", 8: "vegetation", 9: "terrain", 10: "sky", 
    11: "person", 12: "rider", 13: "car", 14: "truck", 15: "bus", 16: "train", 
    17: "motorcycle", 18: "bicycle"
}

LABEL_ID_TO_TRAIN_ID = {
    7: 0, 8: 1, 11: 2, 12: 3, 13: 4, 17: 5, 19: 6, 20: 7, 21: 8, 22: 9, 
    23: 10, 24: 11, 25: 12, 26: 13, 27: 14, 28: 15, 31: 16, 32: 17, 33: 18
}

def compute_conf_matrix(pred, gt, num_classes=19):
    mask = (gt >= 0) & (gt < num_classes)
    label = num_classes * gt[mask].astype('int') + pred[mask]
    count = np.bincount(label, minlength=num_classes**2)
    return count.reshape(num_classes, num_classes)

def evaluate_maskformer_cityscapes_fast():
    print(f"Loading model: {MODEL_ID} on {DEVICE}")
    processor = AutoImageProcessor.from_pretrained(MODEL_ID)
    model = MaskFormerForInstanceSegmentation.from_pretrained(MODEL_ID).to(DEVICE)
    model.eval()

    # Discover images (Matches SegFormer reference)
    search_pattern = os.path.join(IMG_DIR, "*", "*_leftImg8bit.png")
    image_paths = glob.glob(search_pattern)
    
    if not image_paths:
        print(f"No images found in {IMG_DIR}")
        alt_img_dir = os.path.join("cityscapes", "leftImg8bit", "val")
        image_paths = glob.glob(os.path.join(alt_img_dir, "*", "*_leftImg8bit.png"))
        if not image_paths:
            return
        print(f"Found images in alternative path: {alt_img_dir}")

    if MAX_SAMPLES:
        print(f"Limiting evaluation to {MAX_SAMPLES} samples.")
        image_paths = image_paths[:MAX_SAMPLES]

    conf_matrix = np.zeros((19, 19))
    print(f"Starting evaluation on {len(image_paths)} images...")

    for img_path in tqdm(image_paths, desc="Evaluating MaskFormer"):
        filename = os.path.basename(img_path)
        city = os.path.basename(os.path.dirname(img_path))
        file_id = filename.replace("_leftImg8bit.png", "")
        
        if "task_1" in img_path:
            gt_path = os.path.join(GT_DIR, city, f"{file_id}_gtFine_labelIds.png")
        else:
            gt_path = os.path.join("cityscapes", "gtFine", "val", city, f"{file_id}_gtFine_labelIds.png")

        if not os.path.exists(gt_path):
            continue

        image = Image.open(img_path).convert("RGB").resize(EVAL_SIZE, Image.BILINEAR)
        target = Image.open(gt_path).resize(EVAL_SIZE, Image.NEAREST)
        
        target_np = np.array(target)
        new_target = np.full_like(target_np, 255)
        for label_id, train_id in LABEL_ID_TO_TRAIN_ID.items():
            new_target[target_np == label_id] = train_id

        # Inference
        inputs = processor(images=image, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            outputs = model(**inputs)
            # MaskFormer specific post-processing
            pred_labels = processor.post_process_semantic_segmentation(outputs, target_sizes=[EVAL_SIZE[::-1]])[0].cpu().numpy()

        conf_matrix += compute_conf_matrix(pred_labels, new_target, 19)

    # Compute metrics
    true_pos = np.diag(conf_matrix)
    false_pos = np.sum(conf_matrix, axis=0) - true_pos
    false_neg = np.sum(conf_matrix, axis=1) - true_pos
    denominator = true_pos + false_pos + false_neg
    iou = np.divide(true_pos, denominator, out=np.zeros_like(true_pos, dtype=float), where=denominator != 0)
    
    valid_classes = (denominator != 0)
    mean_iou = np.mean(iou[valid_classes])
    pixel_acc = np.sum(true_pos) / np.sum(conf_matrix)

    # --- Report and File Output ---
    output_dir = "evaluation_results"
    os.makedirs(output_dir, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    report_path = os.path.join(output_dir, f"report_maskformer_{timestamp}.txt")

    report_content = [
        "="*40,
        f"Evaluation Results: MaskFormer ({MODEL_ID})",
        f"Date: {datetime.now().isoformat()}",
        "="*40,
        f"Mean IoU: {mean_iou:.4f}",
        f"Overall Accuracy: {pixel_acc:.4f}",
        "-" * 20,
        "Per-class IoU:"
    ]
    for i, score in enumerate(iou):
        label_name = ID2LABEL.get(i, f"Class {i}")
        status = f"{score:.4f}" if denominator[i] > 0 else "N/A"
        report_content.append(f"  {label_name:15}: {status}")
    
    report_text = "\n".join(report_content)
    print("\n" + report_text)
    with open(report_path, "w") as f:
        f.write(report_text)
    print(f"\n[+] Report saved to: {report_path}")

if __name__ == "__main__":
    evaluate_maskformer_cityscapes_fast()


Loading model: facebook/maskformer-resnet101-cityscapes on cuda


The image processor of type `MaskFormerImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/817 [00:00<?, ?it/s]

Starting evaluation on 500 images...


Evaluating MaskFormer: 100%|██████████| 500/500 [01:09<00:00,  7.20it/s]


Evaluation Results: MaskFormer (facebook/maskformer-resnet101-cityscapes)
Date: 2026-03-30T14:12:01.103898
Mean IoU: 0.7598
Overall Accuracy: 0.9542
--------------------
Per-class IoU:
  road           : 0.9776
  sidewalk       : 0.8297
  building       : 0.9143
  wall           : 0.5599
  fence          : 0.5335
  pole           : 0.5931
  traffic light  : 0.6658
  traffic sign   : 0.7532
  vegetation     : 0.9093
  terrain        : 0.6038
  sky            : 0.9417
  person         : 0.7858
  rider          : 0.6187
  car            : 0.9370
  truck          : 0.8065
  bus            : 0.8819
  train          : 0.7391
  motorcycle     : 0.6425
  bicycle        : 0.7430

[+] Report saved to: evaluation_results/report_maskformer_20260330_141201.txt
